# Model Building & Evaluation
### Customer Churn Prediction
---
This notebook covers:
1. Data Preparation (train/test split)
2. Model Building (Logistic Regression, Decision Tree, Random Forest, XGBoost, KNN, SVM)
3. Model Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC)
4. Cross Validation
5. Hyperparameter Tuning
6. Feature Importance
7. Final Model Selection

## 1. Import Libraries

In [ ]:
# Core
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

print("All libraries imported successfully!")

## 2. Load & Prepare Data
> Repeating the EDA preprocessing pipeline to get a clean, model-ready dataset.

In [ ]:
# Load dataset
data = pd.read_excel('P670-dataset.xlsx')
data = data.drop(data.columns[0], axis=1)  # drop index column

# Convert numeric-like object columns
obj_cols = data.select_dtypes(include='object').columns
for col in obj_cols:
    converted = pd.to_numeric(data[col], errors='coerce')
    if converted.notna().mean() > 0.8:
        data[col] = converted

# Fill missing values
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

# Clip outliers (IQR)
for col in num_cols:
    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    IQR = Q3 - Q1
    data[col] = data[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# Encode categorical variables
cat_cols = data.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    data[col] = le.fit_transform(data[col].astype(str))

print(f"Dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

In [ ]:
# Check target variable distribution
print("Target variable - Churn distribution:")
print(data['churn'].value_counts())
print(f"\nChurn Rate: {data['churn'].mean()*100:.2f}%")

plt.figure(figsize=(5,4))
data['churn'].value_counts().plot(kind='bar', color=['#2ecc71','#e74c3c'], edgecolor='black')
plt.title('Churn Distribution')
plt.xlabel('Churn (0=No, 1=Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Train-Test Split

In [ ]:
# Define features (X) and target (y)
X = data.drop('churn', axis=1)
y = data['churn']

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Train-Test Split (80-20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Test set size     : {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

## 4. Model Building
We will train **6 classifiers** and compare their performance.

In [ ]:
# Define models
models = {
    'Logistic Regression'    : LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree'          : DecisionTreeClassifier(random_state=42),
    'Random Forest'          : RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting'      : GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors'    : KNeighborsClassifier(),
    'Support Vector Machine' : SVC(probability=True, random_state=42)
}

# Train all models and collect results
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'y_prob'   : y_prob,
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall'   : recall_score(y_test, y_pred, zero_division=0),
        'f1'       : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_test, y_prob)
    }
    print(f"✅ {name} trained.")

## 5. Model Evaluation

In [ ]:
# Summary comparison table
summary = pd.DataFrame({
    name: {
        'Accuracy' : round(v['accuracy'], 4),
        'Precision': round(v['precision'], 4),
        'Recall'   : round(v['recall'], 4),
        'F1 Score' : round(v['f1'], 4),
        'ROC-AUC'  : round(v['roc_auc'], 4)
    }
    for name, v in results.items()
}).T

print("=" * 75)
print("                  MODEL PERFORMANCE COMPARISON")
print("=" * 75)
print(summary.to_string())
print("=" * 75)

In [ ]:
# Bar chart comparison
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for ax, metric, label in zip(axes, metrics, metric_labels):
    vals = [results[m][metric] for m in results]
    bars = ax.bar(range(len(results)), vals, color=colors, edgecolor='black', alpha=0.85)
    ax.set_xticks(range(len(results)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in results], fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

### 5.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')

for ax, (name, v) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, v['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.tight_layout()
plt.show()

### 5.2 ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))
colors_roc = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for (name, v), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {v['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves - All Models', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Classification Reports

In [ ]:
for name, v in results.items():
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(classification_report(y_test, v['y_pred'], target_names=['No Churn', 'Churn']))

## 6. Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("5-Fold Stratified Cross Validation Results:")
print("-" * 60)

for name, v in results.items():
    scores = cross_val_score(v['model'], X_scaled, y, cv=cv, scoring='f1')
    cv_results[name] = scores
    print(f"{name:<28} F1: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Boxplot of CV scores
plt.figure(figsize=(12, 6))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(figsize=(12, 6), patch_artist=True)
plt.title('Cross-Validation F1 Score Distribution (5-Fold)', fontsize=14, fontweight='bold')
plt.ylabel('F1 Score')
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning (Best Model)
We'll tune the best performing model using GridSearchCV.

In [ ]:
# Identify best model by ROC-AUC
best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f"Best model by ROC-AUC: {best_name} ({results[best_name]['roc_auc']:.4f})")

In [ ]:
# Hyperparameter grid for Random Forest (adjust if your best model differs)
param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest Parameters : {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

print("Tuned Random Forest - Test Set Performance:")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_tuned):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred_tuned):.4f}")
print(f"  F1 Score  : {f1_score(y_test, y_pred_tuned):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_tuned):.4f}")
print()
print(classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churn']))

## 8. Feature Importance

In [ ]:
# Feature importance from tuned Random Forest
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

plt.figure(figsize=(12, 7))
colors_fi = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(importances_sorted))]
importances_sorted.plot(kind='bar', color=colors_fi, edgecolor='black')
plt.title('Feature Importance - Tuned Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=importances_sorted.mean(), color='black', linestyle='--', label='Mean Importance')
plt.legend()
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(importances_sorted.head(10).to_string())

## 9. Final Model Summary

In [ ]:
print("="*60)
print("          FINAL MODEL SELECTION SUMMARY")
print("="*60)

# Full comparison
print("\n📊 All Models - Test Set Performance:")
print(summary.sort_values('ROC-AUC', ascending=False).to_string())

# Winner
print(f"\n🏆 Best Model   : Tuned Random Forest")
print(f"   Best Params  : {grid_search.best_params_}")
print(f"   Accuracy     : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"   F1 Score     : {f1_score(y_test, y_pred_tuned):.4f}")
print(f"   ROC-AUC      : {roc_auc_score(y_test, y_prob_tuned):.4f}")
print()
print("Top 5 Predictive Features:")
for i, (feat, score) in enumerate(importances_sorted.head(5).items(), 1):
    print(f"  {i}. {feat:<30} → {score:.4f}")
print("="*60)

In [ ]:
# Save the final model
import joblib
joblib.dump(best_model, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("✅ Model saved as 'churn_model.pkl'")
print("✅ Scaler saved as 'scaler.pkl'")